# Cold Start Strategies – Impact on Training Performance

## Purpose
This notebook evaluates how different cold start strategies affect downstream
training performance **before any active learning querying is performed**.

We isolate the effect of the initial labeled set by:
- fixing the number of labeled samples,
- disabling active learning cycles,
- using the same model, optimizer, and training procedure.

## Key Question
Does a better cold start (in terms of diversity or u


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.active_learning import ActiveLearningSystem
from config.config import ActiveLearningConfig
from src.utils import save_results_json

## Experimental Setup

All experiments in this notebook share the same configuration:

- Same dataset and validation split
- Same model architecture (U-Net)
- Same optimizer and hyperparameters
- Same number of training epochs
- Same random seed

The only difference between experiments is **which samples are selected
as the initial labeled set** by the cold start strategy.


In [ ]:
# Load cold start selections generated in the previous notebook
with open("../results/cold_start_selections.json", "r") as f:
    cold_start_selections = json.load(f)

list(cold_start_selections.keys())

## Training Protocol

For each cold start strategy:

1. Initialize an `ActiveLearningSystem`
2. Override the labeled pool using the selected indices
3. Disable active learning cycles (no querying)
4. Train the model for a fixed number of epochs
5. Record validation metrics at each epoch

This ensures that performance differences are attributable **only**
to the quality of the initial labeled set.


In [ ]:
BASE_CONFIG_PATH = "../experiments/configs/example_segmentation.yaml"
EPOCHS = 15
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
results = {}

for strategy, labeled_indices in cold_start_selections.items():
    print(f"\n{'='*60}")
    print(f"Training with cold start strategy: {strategy}")
    print(f"{'='*60}")

    # Load configuration
    config = ActiveLearningConfig.from_yaml(BASE_CONFIG_PATH)
    config.al_cycles = 0                 # Disable querying
    config.use_wandb = False
    config.seed = SEED

    # Initialize system (skip internal cold start)
    al_system = ActiveLearningSystem(config, skip_cold_start=True)

    # Override labeled pool
    al_system.set_labeled_indices(labeled_indices)

    # Train only
    al_system.train(epochs=EPOCHS)

    # Store results
    results[strategy] = {
        "history": al_system.history,
        "final_dice": al_system.history["val_dice"][-1],
        "final_iou": al_system.history["val_iou"][-1],
        "num_labeled": len(labeled_indices),
    }


## Final Performance Comparison

We compare cold start strategies using:
- Final Dice score
- Final mean IoU
- Number of labeled samples (fixed)

Higher Dice and IoU indicate better segmentation performance.

In [ ]:
summary = []

for strategy, res in results.items():
    summary.append({
        "strategy": strategy,
        "final_dice": res["final_dice"],
        "final_iou": res["final_iou"],
        "num_labeled": res["num_labeled"],
    })

df_summary = pd.DataFrame(summary).sort_values("final_dice", ascending=False)
df_summary

## Learning Curves

Training dynamics provide insight into:
- convergence speed,
- stability of optimization,
- usefulness of the initial labeled set.

Faster convergence and higher plateaus indicate a more effective cold start.

In [ ]:
plt.figure(figsize=(8, 5))

for strategy, res in results.items():
    dice_curve = res["history"]["val_dice"]
    plt.plot(range(1, len(dice_curve) + 1), dice_curve, label=strategy)

plt.xlabel("Epoch")
plt.ylabel("Validation Dice")
plt.title("Impact of Cold Start Strategy on Training")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

This experiment allows us to identify which cold start strategies:

- lead to faster performance gains,
- reach higher final accuracy,
- provide a stable optimization trajectory.

The best-performing strategy will be used as the default cold start
configuration in the full active learning experiments (Notebook 04).


In [ ]:
for strategy, res in results.items():
    save_results_json(
        results=res,
        experiment_name=f"cold_start_training_{strategy}",
        results_dir="../results"
    )

## Takeaway

Cold start selection has a measurable impact on downstream training performance.

By reusing the same training pipeline used for active learning, this analysis:
- avoids duplicated logic,
- ensures consistency across experiments,
- provides a principled basis for selecting a cold start strategy.

The selected strategy will be fixed in subsequent active learning experiments.
